<a href="https://colab.research.google.com/github/maggoatt/Grounded-Text-Summarization-of-Research-Papers/blob/main/PaperEmbeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paper Embeddings Generation

Creates and saves embeddings for chunks of text from original papers.

Author: Lawrence Zhou



In [ ]:
!unzip data.zip

In [ ]:
from pathlib import Path
import json
from sentence_transformers import SentenceTransformer
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

papers_dir = Path("../../../data")


In [ ]:
import numpy as np

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
def chunk(text, max_words = 200, overlap_sentences = 2):
  """
  This function will chunk a paper by separating each paper into sentences, then
  placing a max amount of sentences/words in each chunk because the model has
  a token limit. There will also be two overlapping sentences in order to carry
  import info from the previous sections over. Indicies array will keep track of
  what chunks contain which sentences.
  """
  sentences = sent_tokenize(text)
  chunks = []
  indices = []

  i = 0
  while i < len(sentences):
    curr_chunk = []
    curr_word_count = 0
    j = i
    while j < len(sentences):
      curr_word_count += len(sentences[j].split())
      if curr_word_count > max_words and curr_chunk:
        break
      curr_chunk.append(sentences[j])
      j += 1
    chunks.append(" ".join(curr_chunk))
    indices.append((i, j - 1))
    if (j - i > overlap_sentences):
      i = j - overlap_sentences
      # in case one sentence is just too long or yoou're going backwards
    else:
      i = j
  return chunks, indices

In [ ]:
import os
os.makedirs("embeddings", exist_ok=True)

In [ ]:
i = 0
for file in papers_dir.glob("*.json"):
    paper_id = file.stem
    paper = json.loads(file.read_text(encoding="utf-8"))
    title = paper.get("title")
    text = " ".join(section["text"] for section in paper["sections"])
    file_chunks, file_indicies = chunk(text)
    embeddings = model.encode(file_chunks, show_progress_bar=True)
    print(embeddings.shape)
    np.save(f"../../../embeddings/{paper_id}_embeddings.npy", embeddings)
    metadata = {
        "chunks": file_chunks,
        "file_indicies": file_indicies
    }
    with open(f"../../../embeddings/{paper_id}_metadata.json", "w") as f:
      json.dump(metadata, f)